# 🧹 02 — Data Cleaning & Preprocessing
Run cleaning functions and save cleaned CSVs to `data/processed/`.
This notebook calls functions from `src/preprocessing/clean_data.py` if available.

In [ ]:
import os
import pandas as pd
from pathlib import Path

RAW = Path('..') / 'data' / 'raw'
PROCESSED = Path('..') / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

# Load raw
perf = pd.read_csv(RAW / 'performance.csv')
values = pd.read_csv(RAW / 'transfermarkt_values.csv')
tweets = pd.read_csv(RAW / 'tweets.csv')
inj = pd.read_csv(RAW / 'injuries.csv')

print('Raw data loaded')

## Clean Transfermarkt values

In [ ]:
vals = values.copy()
vals.columns = [c.lower().strip() for c in vals.columns]
vals['market_value_eur'] = pd.to_numeric(vals['market_value_eur'], errors='coerce')
vals = vals.drop_duplicates(subset=['player_name'])
vals.to_csv(PROCESSED / 'clean_transfermarkt.csv', index=False)
display(vals.head())

## Clean performance (StatsBomb aggregated sample)

In [ ]:
perf_c = perf.copy()
perf_c.columns = [c.lower().strip() for c in perf_c.columns]
for c in ['shots','passes','tackles','events']:
    if c in perf_c.columns:
        perf_c[c] = pd.to_numeric(perf_c[c], errors='coerce').fillna(0)
perf_c.to_csv(PROCESSED / 'clean_statsbomb.csv', index=False)
display(perf_c.head())

## Clean tweets → aggregate sentiment per player

In [ ]:
tw = tweets.copy()
tw.columns = [c.lower().strip() for c in tw.columns]
tw['tweet_date'] = pd.to_datetime(tw['tweet_date'], errors='coerce')
sent = tw.groupby('player_keyword')['vader_compound'].mean().reset_index()
sent = sent.rename(columns={'player_keyword':'player_name','vader_compound':'sentiment_score'})
sent.to_csv(PROCESSED / 'clean_sentiment.csv', index=False)
display(sent.head())

## Clean injuries

In [ ]:
inj_c = inj.copy()
inj_c.columns = [c.lower().strip() for c in inj_c.columns]
for col in ['start_date','end_date']:
    if col in inj_c.columns:
        inj_c[col] = pd.to_datetime(inj_c[col], errors='coerce')
if 'days_absent' not in inj_c.columns:
    if all(x in inj_c.columns for x in ('start_date','end_date')):
        inj_c['days_absent'] = (inj_c['end_date'] - inj_c['start_date']).dt.days.fillna(0).astype(int)
    else:
        inj_c['days_absent'] = 0
inj_agg = inj_c.groupby('player_name', as_index=False)['days_absent'].sum()
inj_agg.to_csv(PROCESSED / 'clean_injuries.csv', index=False)
display(inj_agg.head())

## ✅ All cleaned files saved under `data/processed/`